# SurfaceTarget-CRC — phase 02 — patch 2 3

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 2 — PATCH 2.3
#
# CPTAC CROSS-MATRIX QUANTIFICATION AUDIT
#
# Objectives:
# 1. Validate CPTAC source matrices
# 2. Harmonize tumor and normal patient identifiers
# 3. Compare derived differences with reported paired ratios
# 4. Evaluate global and patient-level concordance
# 5. Examine CDH3, CLDN1 and other leading genes
# 6. Save audit tables, figure and provenance
#
# IMPORTANT:
# - No new protein differential expression tests
# - No imputation
# - No automatic acceptance of derived ratios
# - No source data modification
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import spearmanr

import hashlib
import json
import re


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
RAW = ROOT / "data/raw/cptac_colon"
RESULTS = ROOT / "results/tables"
FIGURES = ROOT / "results/figures"

RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

SOURCE_PATH = (
    META / "cptac_colon_source_manifest.csv"
)

CLINICAL_PATH = (
    META / "cptac_colon_paired_sample_metadata.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 2 — PATCH 2.3")
print("CPTAC CROSS-MATRIX QUANTIFICATION AUDIT")
print("=" * 65)


# ------------------------------------------------------------
# 2. Validate source files
# ------------------------------------------------------------

source = pd.read_csv(
    SOURCE_PATH,
    dtype=str,
    keep_default_na=False
)

if source["dataset"].duplicated().any():
    raise RuntimeError(
        "Duplicate CPTAC source datasets."
    )


def calculate_sha256(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


source_paths = {}

for _, row in source.iterrows():

    path = RAW / row["filename"]

    if not path.exists():
        raise FileNotFoundError(path)

    if calculate_sha256(path) != row["sha256"]:
        raise RuntimeError(
            f"Checksum mismatch: {row['dataset']}"
        )

    source_paths[row["dataset"]] = path

print("\nSource checksum validation: PASSED")


# ------------------------------------------------------------
# 3. Load proteomics matrices
# ------------------------------------------------------------

def load_matrix(dataset):

    df = pd.read_csv(
        source_paths[dataset],
        sep="\t",
        index_col=0,
        na_values=["NA", "N/A", ""],
        low_memory=False
    )

    df.index = (
        df.index.astype(str).str.strip()
    )

    df.columns = (
        df.columns.astype(str).str.strip()
    )

    if df.index.duplicated().any():
        raise RuntimeError(
            f"Duplicate gene symbols: {dataset}"
        )

    if df.columns.duplicated().any():
        raise RuntimeError(
            f"Duplicate sample identifiers: {dataset}"
        )

    numeric = df.apply(
        pd.to_numeric,
        errors="raise"
    )

    if np.isinf(
        numeric.to_numpy(dtype=float)
    ).any():
        raise RuntimeError(
            f"Infinite values: {dataset}"
        )

    return numeric


tumor = load_matrix("tumor_proteomics")

normal = load_matrix("normal_proteomics")

paired = load_matrix("paired_proteomics")


print("\nMatrices loaded.")

print("Tumor:", tumor.shape)
print("Normal:", normal.shape)
print("Reported paired:", paired.shape)


# ------------------------------------------------------------
# 4. Harmonize patient identifiers
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PATIENT IDENTIFIER VALIDATION")
print("-" * 65)


PATTERN = re.compile(
    r"^(\d{2}CO\d{3})(N)?$"
)


def build_patient_map(columns, dataset):

    mapping = {}

    for sample_id in columns:

        match = PATTERN.fullmatch(
            str(sample_id)
        )

        if match is None:
            raise RuntimeError(
                f"Unrecognized {dataset} identifier: "
                f"{sample_id}"
            )

        patient_id = match.group(1)
        normal_suffix = match.group(2)

        if dataset == "Normal":

            if normal_suffix != "N":
                raise RuntimeError(
                    f"Unexpected normal label: {sample_id}"
                )

        else:

            if normal_suffix is not None:
                raise RuntimeError(
                    f"Unexpected {dataset} label: {sample_id}"
                )

        if patient_id in mapping:
            raise RuntimeError(
                f"Duplicate patient: {patient_id}"
            )

        mapping[patient_id] = sample_id

    return mapping


tumor_map = build_patient_map(
    tumor.columns,
    "Tumor"
)

normal_map = build_patient_map(
    normal.columns,
    "Normal"
)

paired_map = build_patient_map(
    paired.columns,
    "Paired"
)


shared_patients = sorted(
    set(tumor_map)
    & set(normal_map)
    & set(paired_map)
)


print("\nPatients in all three matrices:",
      len(shared_patients))


if len(shared_patients) != 96:
    raise RuntimeError(
        "Expected 96 fully matched CPTAC patients."
    )


print("Patient correspondence: PASSED")


# ------------------------------------------------------------
# 5. Identify comparable protein rows
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PROTEIN IDENTIFIER CORRESPONDENCE")
print("-" * 65)


common_genes = (
    paired.index
    .intersection(tumor.index)
    .intersection(normal.index)
)


print("\nTumor proteins:", len(tumor))
print("Normal proteins:", len(normal))
print("Reported paired proteins:", len(paired))

print(
    "Proteins represented in all three:",
    len(common_genes)
)


if len(common_genes) < 3000:
    raise RuntimeError(
        "Unexpectedly low protein overlap."
    )


# ------------------------------------------------------------
# 6. Align measurements
# ------------------------------------------------------------

tumor_columns = [
    tumor_map[p]
    for p in shared_patients
]

normal_columns = [
    normal_map[p]
    for p in shared_patients
]

paired_columns = [
    paired_map[p]
    for p in shared_patients
]


tumor_values = (
    tumor.loc[
        common_genes,
        tumor_columns
    ]
    .to_numpy(dtype=float)
)

normal_values = (
    normal.loc[
        common_genes,
        normal_columns
    ]
    .to_numpy(dtype=float)
)

reported_values = (
    paired.loc[
        common_genes,
        paired_columns
    ]
    .to_numpy(dtype=float)
)


# Diagnostic subtraction only.
#
# We are NOT yet assuming the different
# matrices have equivalent normalization.

derived_values = (
    tumor_values - normal_values
)


valid = (
    np.isfinite(derived_values)
    & np.isfinite(reported_values)
)


n_comparable = int(valid.sum())


print("\n" + "-" * 65)
print("CROSS-MATRIX MEASUREMENT AVAILABILITY")
print("-" * 65)


print(
    "\nPotential gene-patient comparisons:",
    int(derived_values.size)
)

print(
    "Complete comparisons:",
    n_comparable
)

print(
    "Complete comparison percentage:",
    round(
        100 * n_comparable / derived_values.size,
        2
    )
)


if n_comparable < 5000:
    raise RuntimeError(
        "Insufficient complete measurements."
    )


# ------------------------------------------------------------
# 7. Global concordance statistics
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GLOBAL QUANTIFICATION CONCORDANCE")
print("-" * 65)


derived_observed = derived_values[valid]

reported_observed = reported_values[valid]


difference = (
    derived_observed - reported_observed
)


pearson = float(
    np.corrcoef(
        derived_observed,
        reported_observed
    )[0, 1]
)


spearman = float(
    spearmanr(
        derived_observed,
        reported_observed
    ).statistic
)


median_absolute_error = float(
    np.median(
        np.abs(difference)
    )
)


median_difference = float(
    np.median(difference)
)


p95_absolute_error = float(
    np.percentile(
        np.abs(difference),
        95
    )
)


print("\nComplete observations:", n_comparable)

print(
    "Pearson correlation:",
    round(pearson, 5)
)

print(
    "Spearman correlation:",
    round(spearman, 5)
)

print(
    "Median derived-minus-reported:",
    round(median_difference, 5)
)

print(
    "Median absolute difference:",
    round(median_absolute_error, 5)
)

print(
    "95th percentile absolute difference:",
    round(p95_absolute_error, 5)
)


print("\nIMPORTANT:")
print(
    "Correlation alone does not establish "
    "measurement-scale equivalence."
)


# ------------------------------------------------------------
# 8. Patient-level comparison
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PATIENT-LEVEL CONCORDANCE")
print("-" * 65)


patient_records = []


for column_index, patient_id in enumerate(
    shared_patients
):

    x = derived_values[:, column_index]

    y = reported_values[:, column_index]

    mask = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    n = int(mask.sum())

    if n >= 50:

        px = x[mask]
        py = y[mask]

        correlation = float(
            np.corrcoef(px, py)[0, 1]
        )

        median_error = float(
            np.median(
                np.abs(px - py)
            )
        )

        median_offset = float(
            np.median(px - py)
        )

    else:

        correlation = np.nan
        median_error = np.nan
        median_offset = np.nan

    patient_records.append({

        "patient_id": patient_id,

        "n_complete_proteins": n,

        "pearson_correlation":
            correlation,

        "median_absolute_difference":
            median_error,

        "median_derived_minus_reported":
            median_offset

    })


patient_audit = pd.DataFrame(
    patient_records
)


print("\nPatient-level summary:")

print(
    patient_audit[
        [
            "n_complete_proteins",
            "pearson_correlation",
            "median_absolute_difference",
            "median_derived_minus_reported"
        ]
    ]
    .describe()
    .round(4)
    .to_string()
)


# ------------------------------------------------------------
# 9. Load MSS clinical annotations
# ------------------------------------------------------------

clinical = pd.read_csv(
    CLINICAL_PATH,
    dtype=str,
    keep_default_na=False
)


id_columns = [
    c for c in [
        "cptac_patient_id",
        "index",
        "Unnamed: 0"
    ]
    if c in clinical.columns
]


if len(id_columns) != 1:
    raise RuntimeError(
        "Clinical patient ID column is ambiguous."
    )


clinical = clinical.rename(
    columns={
        id_columns[0]: "cptac_patient_id"
    }
)


if clinical["cptac_patient_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate clinical patient IDs."
    )


clinical = clinical.set_index(
    "cptac_patient_id"
)


msi_status = (
    clinical.loc[
        shared_patients,
        "msi_classification"
    ]
    .astype(str)
    .str.strip()
    .str.upper()
)


mss_patients = [
    p for p in shared_patients
    if msi_status.loc[p] == "MSS"
]


print("\nPublished MSS matched patients:",
      len(mss_patients))


if len(mss_patients) != 73:
    raise RuntimeError(
        "Expected 73 published MSS patients."
    )


# ------------------------------------------------------------
# 10. Examine leading genes
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING GENE MEASUREMENT AUDIT")
print("-" * 65)


TARGET_GENES = [
    "CDH3",
    "CLDN1",
    "KRT80",
    "CEMIP",
    "INHBA",
    "WNT2",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "CTHRC1",
    "S100P"
]


mss_tumor_columns = [
    tumor_map[p]
    for p in mss_patients
]

mss_normal_columns = [
    normal_map[p]
    for p in mss_patients
]

mss_paired_columns = [
    paired_map[p]
    for p in mss_patients
]


gene_records = []


for gene in TARGET_GENES:

    in_tumor = gene in tumor.index
    in_normal = gene in normal.index
    in_reported = gene in paired.index

    n_complete_separate = 0

    derived_median = np.nan
    derived_fraction_positive = np.nan

    reported_n = 0
    reported_median = np.nan

    if in_tumor and in_normal:

        t = tumor.loc[
            gene,
            mss_tumor_columns
        ].to_numpy(dtype=float)

        n = normal.loc[
            gene,
            mss_normal_columns
        ].to_numpy(dtype=float)

        complete = (
            np.isfinite(t)
            & np.isfinite(n)
        )

        n_complete_separate = int(
            complete.sum()
        )

        if n_complete_separate > 0:

            delta = (
                t[complete] - n[complete]
            )

            derived_median = float(
                np.median(delta)
            )

            derived_fraction_positive = float(
                np.mean(delta > 0)
            )

    if in_reported:

        r = paired.loc[
            gene,
            mss_paired_columns
        ].to_numpy(dtype=float)

        r = r[np.isfinite(r)]

        reported_n = len(r)

        if reported_n > 0:

            reported_median = float(
                np.median(r)
            )

    gene_records.append({

        "gene_name": gene,

        "in_tumor_matrix": in_tumor,

        "in_normal_matrix": in_normal,

        "in_reported_paired_matrix": in_reported,

        "mss_complete_separate_pairs":
            n_complete_separate,

        "diagnostic_derived_median_log2_difference":
            derived_median,

        "diagnostic_derived_fraction_positive":
            derived_fraction_positive,

        "reported_paired_mss_measurements":
            reported_n,

        "reported_paired_median_log2ratio":
            reported_median

    })


gene_audit = pd.DataFrame(
    gene_records
)


print("\n")

print(
    gene_audit.to_string(
        index=False,
        float_format=lambda x: f"{x:.4g}"
    )
)


print("\nCAUTION:")
print(
    "Derived differences are diagnostic only. "
    "They are not yet accepted as validated "
    "tumor-normal protein enrichment measurements."
)


# ------------------------------------------------------------
# 11. Generate concordance figure
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING CROSS-MATRIX FIGURE")
print("-" * 65)


rng = np.random.default_rng(42)

n_plot = min(
    15000,
    len(derived_observed)
)

plot_indices = rng.choice(
    len(derived_observed),
    size=n_plot,
    replace=False
)


x_plot = reported_observed[
    plot_indices
]

y_plot = derived_observed[
    plot_indices
]


fig, ax = plt.subplots(
    figsize=(8, 7)
)


ax.scatter(
    x_plot,
    y_plot,
    s=6,
    alpha=0.25
)


limits = [
    float(min(
        np.min(x_plot),
        np.min(y_plot)
    )),
    float(max(
        np.max(x_plot),
        np.max(y_plot)
    ))
]


ax.plot(
    limits,
    limits,
    linestyle="--",
    linewidth=1,
    label="Exact agreement"
)


ax.set_xlabel(
    "Reported paired tumor-normal log2 ratio"
)

ax.set_ylabel(
    "Tumor minus normal (separate log-ratio matrices)"
)

ax.set_title(
    "CPTAC Colon — Cross-Matrix Quantification Audit"
)

ax.grid(alpha=0.2)

ax.legend()

fig.tight_layout()


figure_path = (
    FIGURES /
    "cptac_crossmatrix_ratio_concordance.png"
)


fig.savefig(
    figure_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


print("\nFigure saved.")


# ------------------------------------------------------------
# 12. Save audit tables
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING CROSS-MATRIX AUDIT")
print("-" * 65)


patient_audit.to_csv(

    RESULTS /
    "cptac_crossmatrix_patient_concordance.csv",

    index=False

)


gene_audit.to_csv(

    RESULTS /
    "cptac_crossmatrix_leading_gene_review.csv",

    index=False

)


# ------------------------------------------------------------
# 13. Save provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "2.3",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "CPTAC Colon via LinkedOmics",

    "analysis": (
        "Diagnostic concordance between "
        "reported paired protein log2 ratios "
        "and differences derived from separate "
        "tumor and normal log-ratio matrices"
    ),

    "matched_patients": int(
        len(shared_patients)
    ),

    "published_mss_patients": int(
        len(mss_patients)
    ),

    "common_proteins": int(
        len(common_genes)
    ),

    "complete_comparisons": n_comparable,

    "global_pearson": pearson,

    "global_spearman": spearman,

    "median_derived_minus_reported":
        median_difference,

    "median_absolute_difference":
        median_absolute_error,

    "p95_absolute_difference":
        p95_absolute_error,

    "diagnostic_subtraction_performed": True,

    "derived_ratios_accepted_for_inference": False,

    "additional_protein_differential_expression":
        False,

    "imputation_performed": False,

    "limitations": [

        "The separate matrices may have "
        "different quantification or "
        "normalization characteristics.",

        "Agreement between datasets does not "
        "independently establish biological validity.",

        "Missing proteomic measurements "
        "may be nonrandom.",

        "Derived differences are not yet "
        "approved for formal enrichment inference.",

        "Protein abundance does not establish "
        "cell-surface accessibility."

    ]
}


with open(
    META / "cptac_crossmatrix_audit_provenance.json",
    "w"
) as f:

    json.dump(
        provenance,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 14. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 2.3 SUMMARY")
print("=" * 65)


print("\nMatched patients:", len(shared_patients))

print("Published MSS patients:", len(mss_patients))

print("Comparable proteins:", len(common_genes))

print("Complete measurement comparisons:",
      n_comparable)


print("\nGlobal Pearson:", round(pearson, 5))

print("Global Spearman:", round(spearman, 5))

print(
    "Median absolute difference:",
    round(median_absolute_error, 5)
)


print("\nSaved outputs:")

print("1. cptac_crossmatrix_patient_concordance.csv")

print("2. cptac_crossmatrix_leading_gene_review.csv")

print("3. cptac_crossmatrix_audit_provenance.json")

print("4. cptac_crossmatrix_ratio_concordance.png")


print("\nNo new protein enrichment tests performed.")

print("No source measurements modified.")

print("No additional datasets downloaded.")


print("\nPATCH 2.3 COMPLETE")

print("=" * 65)